# Fine-tune Laya for gutcheck's prompt-guard pack

Runs `gutcheck finetune` on a Kaggle T4, pushes the checkpoint to your Hugging Face account, then
evaluates it on the pack's test sets.

Before you run it:

1. **Settings → Accelerator:** GPU T4 (one GPU is used).
2. **Settings → Internet:** on.
3. **Add-ons → Secrets:** add `HF_TOKEN`, a Hugging Face token with write access.
4. Set `HF_REPO` below, then **Run all**. Training takes roughly 15–30 minutes.

At the end, copy the repo id and commit it prints.

In [ ]:
HF_REPO = "your-username/laya-prompt-guard"  # created if it doesn't exist
BASE = "english"  # or "multilingual"
EPOCHS = 4
GUTCHECK_REF = "master"  # branch, tag or commit of gutcheck to install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "gutcheck[eval] @ git+https://github.com/16SULPHUR/gutcheck@{GUTCHECK_REF}"
!gutcheck --version

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
OUT = "/kaggle/working/laya-prompt-guard"

## Train and push

Each question trains on its dataset's training split, minus a 20% held-out slice that measures the
result and fits the new temperature. The held-out rows are uploaded with the model so gutcheck can
recalibrate from them later.

In [ ]:
args = f"--base {BASE} --epochs {EPOCHS} --device cuda --out {OUT} --push {HF_REPO}"
!gutcheck finetune prompt-guard {args}

## Evaluate on the test sets

Runs the fine-tuned pack (`prompt-guard@2`, written to `OUT/pack`) against the same test splits as
the bundled pack, so the numbers compare directly with
[EVAL.md](https://github.com/16SULPHUR/gutcheck/blob/master/src/gutcheck/packs/prompt-guard/EVAL.md).

In [ ]:
from pathlib import Path

config = f"packs: {{dirs: [{OUT}/pack]}}\nengine: {{device: cuda, models: []}}\n"
Path("/kaggle/working/gutcheck.yaml").write_text(config)
!gutcheck eval prompt-guard@2 --config /kaggle/working/gutcheck.yaml --write

In [ ]:
import json

from huggingface_hub import HfApi

info = HfApi().model_info(HF_REPO)
print(f"repo:   {HF_REPO}\ncommit: {info.sha}")
print(json.dumps(json.loads(Path(OUT, "training.json").read_text())["questions"], indent=2))